# Bronze Layer: Automated Data Validation & Quality Gate Notebook
Performs automated Source-to-Target data reconciliation between MySQL source and Delta Lake Bronze tables:
1. **Row Count Reconciliation**: Validates source row count matches target Delta row count.
2. **Null Value Check**: Ensures Primary Keys / Merge Keys contain zero NULL values.
3. **Duplicate Check**: Ensures Primary Keys / Merge Keys are 100% unique without duplicates.

Audit results are appended to `spark_training.metadata_schema.bronze_validation_log`. If any check fails, the notebook raises a descriptive Exception to trigger native Databricks Workflow email notifications.

In [ ]:
# 1. Install Driver Libraries for Serverless / Community Compute
%pip install pymysql cryptography --quiet

In [ ]:
%run ../../src/utilities/logger

In [ ]:
%run ../../src/utilities/utils

In [ ]:
# 3. Initialize SparkSession and Task Logger
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
import pymysql
import pandas as pd

# Ensure SparkSession is active
spark = SparkSession.builder.getOrCreate()

# Initialize logger for this validation notebook
logger = get_task_logger(notebook_name_override="bronze_validation")
logger.info("=== Starting Bronze Data Validation Pipeline ===")

In [ ]:
# 4. Interactive Widgets for Dynamic Tunnel Port & Batch ID
dbutils.widgets.text("tunnel_port", "30577", "Bore Tunnel Port")
db_port = dbutils.widgets.get("tunnel_port")

dbutils.widgets.text("batch_id", "ALL", "Batch ID (1, 2, 3, 4 or ALL)")
batch_id = dbutils.widgets.get("batch_id").strip()

logger.info(f"Active Tunnel Port: {db_port} | Target Batch ID: {batch_id}")
print(f"Active Tunnel Port: {db_port} | Target Batch ID: {batch_id}")

In [ ]:
# 5. Retrieve Credentials Securely from Databricks Secrets
SECRET_SCOPE = "wanderbricks_scope"

db_user = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_user")
db_password = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_password")
db_host = dbutils.secrets.get(scope=SECRET_SCOPE, key="tunnel_host")
db_name = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_db")

logger.info(f"Retrieved credentials from scope '{SECRET_SCOPE}' for user '{db_user}'")
print(f"Target database: {db_name} at host: {db_host}")

In [ ]:
# 6. Ensure Validation Audit Log Table Exists in Unity Catalog
create_audit_table_sql = """
CREATE TABLE IF NOT EXISTS spark_training.metadata_schema.bronze_validation_log (
    validation_id BIGINT GENERATED ALWAYS AS IDENTITY,
    table_id INT,
    table_name STRING,
    batch_id INT,
    check_type STRING,
    column_checked STRING,
    source_value STRING,
    target_value STRING,
    status STRING,
    execution_time TIMESTAMP,
    comment STRING
);
"""
spark.sql(create_audit_table_sql)
logger.info("Ensured 'spark_training.metadata_schema.bronze_validation_log' exists.")

In [ ]:
# 7. Automated Source-to-Target Data Quality & Reconciliation Engine
# Fetch active tables configured for the Bronze layer and selected batch
active_tables = get_active_tables(target_layer="bronze", batch_id=batch_id)

# Python-level defensive filter in case batch_id was not yet filtered in SQL
if batch_id and batch_id.upper() != "ALL":
    active_tables = [t for t in active_tables if str(t.get("batch_id", "")) == str(batch_id)]
    logger.info(f"Filtered for Batch ID '{batch_id}': Found {len(active_tables)} active tables to validate.")
    print(f"Filtered for Batch ID '{batch_id}': Found {len(active_tables)} active Bronze tables to validate.\n")
else:
    logger.info(f"Found {len(active_tables)} active tables for Bronze validation (All Batches).")
    print(f"Found {len(active_tables)} active Bronze tables to validate (All Batches).\n")

if not active_tables:
    logger.warning(f"No active tables found matching Batch ID '{batch_id}'. Skipping validation loop.")
    print(f"No active tables found matching Batch ID '{batch_id}'.")

validation_records = []

# Connect to MySQL once for all source row count checks
connection = None
try:
    connection = pymysql.connect(
        host=db_host,
        port=int(db_port),
        user=db_user,
        password=db_password,
        database=db_name,
        connect_timeout=15
    )
    cursor = connection.cursor()
    
    for metadata in active_tables:
        table_id = metadata["table_id"]
        source_table_name = metadata["source_table_name"]
        target_table_name = metadata["target_table_name"]
        target_table = f"{metadata['target_database_name']}.{metadata['target_schema_name']}.{metadata['target_table_name']}"
        merge_key = metadata.get("merge_key")
        batch_num = metadata.get("batch_id", 1)
        
        logger.info(f"------------------------------------------------------------")
        logger.info(f"Validating Table ID {table_id} [Batch {batch_num}]: '{source_table_name}' vs '{target_table}'")
        print(f"\nValidating Table {table_id} [Batch {batch_num}]: {source_table_name} -> {target_table}")
        
        # Check if Delta target table exists
        table_exists = spark.catalog.tableExists(target_table)
        if not table_exists:
            comment_msg = f"Target table '{target_table}' does not exist in Unity Catalog!"
            logger.error(comment_msg)
            print(f"  ❌ FAIL: {comment_msg}")
            validation_records.append({
                "table_id": table_id,
                "table_name": target_table_name,
                "batch_id": int(batch_num) if batch_num else None,
                "check_type": "TABLE_EXISTENCE",
                "column_checked": "TABLE",
                "source_value": "EXISTS",
                "target_value": "NOT_FOUND",
                "status": "FAIL",
                "comment": comment_msg
            })
            continue

        # ============================================================
        # CHECK 1: ROW COUNT RECONCILIATION
        # ============================================================
        try:
            cursor.execute(f"SELECT COUNT(*) FROM {source_table_name};")
            source_row_count = cursor.fetchone()[0]
            target_row_count = spark.sql(f"SELECT COUNT(*) FROM {target_table};").collect()[0][0]
            
            if source_row_count == target_row_count:
                rc_status = "PASS"
                rc_comment = f"Row count perfectly matched: {target_row_count} rows."
                print(f"  ✅ [ROW_COUNT] PASS: Source ({source_row_count}) == Target ({target_row_count})")
            else:
                rc_status = "FAIL"
                diff = abs(source_row_count - target_row_count)
                rc_comment = f"Row count mismatch! Source: {source_row_count}, Target: {target_row_count} (Diff: {diff})."
                print(f"  ❌ [ROW_COUNT] FAIL: Source ({source_row_count}) != Target ({target_row_count})")
            
            validation_records.append({
                "table_id": table_id,
                "table_name": target_table_name,
                "batch_id": int(batch_num) if batch_num else None,
                "check_type": "ROW_COUNT",
                "column_checked": "ALL",
                "source_value": str(source_row_count),
                "target_value": str(target_row_count),
                "status": rc_status,
                "comment": rc_comment
            })
        except Exception as e:
            logger.error(f"Row count check failed for {source_table_name}: {e}")
            validation_records.append({
                "table_id": table_id,
                "table_name": target_table_name,
                "batch_id": int(batch_num) if batch_num else None,
                "check_type": "ROW_COUNT",
                "column_checked": "ALL",
                "source_value": "ERROR",
                "target_value": "ERROR",
                "status": "FAIL",
                "comment": f"Error executing row count check: {str(e)[:250]}"
            })

        # ============================================================
        # CHECK 2: NULL VALUE CHECK ON PRIMARY / MERGE KEYS
        # ============================================================
        if merge_key:
            keys = [k.strip() for k in merge_key.split(",") if k.strip()]
            null_conditions = " OR ".join([f"{k} IS NULL" for k in keys])
            try:
                null_query = f"SELECT COUNT(*) FROM {target_table} WHERE {null_conditions};"
                null_count = spark.sql(null_query).collect()[0][0]
                
                if null_count == 0:
                    null_status = "PASS"
                    null_comment = f"Zero NULLs found in primary/merge key [{merge_key}]."
                    print(f"  ✅ [NULL_CHECK] PASS: 0 nulls in primary key [{merge_key}]")
                else:
                    null_status = "FAIL"
                    null_comment = f"Found {null_count} rows with NULL in primary/merge key [{merge_key}]!"
                    print(f"  ❌ [NULL_CHECK] FAIL: {null_count} nulls in primary key [{merge_key}]")
                
                validation_records.append({
                    "table_id": table_id,
                    "table_name": target_table_name,
                    "batch_id": int(batch_num) if batch_num else None,
                    "check_type": "NULL_CHECK",
                    "column_checked": merge_key,
                    "source_value": "0",
                    "target_value": str(null_count),
                    "status": null_status,
                    "comment": null_comment
                })
            except Exception as e:
                logger.error(f"Null check error for {target_table}: {e}")
                validation_records.append({
                    "table_id": table_id,
                    "table_name": target_table_name,
                    "batch_id": int(batch_num) if batch_num else None,
                    "check_type": "NULL_CHECK",
                    "column_checked": merge_key,
                    "source_value": "0",
                    "target_value": "ERROR",
                    "status": "FAIL",
                    "comment": f"Error executing null check: {str(e)[:250]}"
                })

        # ============================================================
        # CHECK 3: DUPLICATION CHECK ON PRIMARY / MERGE KEYS
        # ============================================================
        if merge_key:
            keys = [k.strip() for k in merge_key.split(",") if k.strip()]
            keys_str = ", ".join(keys)
            try:
                dup_query = f"""
                    SELECT COUNT(*) FROM (
                        SELECT {keys_str}, COUNT(*) AS count_val
                        FROM {target_table}
                        GROUP BY {keys_str}
                        HAVING COUNT(*) > 1
                    )
                """
                dup_count = spark.sql(dup_query).collect()[0][0]
                
                if dup_count == 0:
                    dup_status = "PASS"
                    dup_comment = f"Zero duplicate keys found on [{merge_key}]."
                    print(f"  ✅ [DUPLICATE_CHECK] PASS: 0 duplicate keys on [{merge_key}]")
                else:
                    dup_status = "FAIL"
                    dup_comment = f"Found {dup_count} duplicate primary key sets on [{merge_key}]!"
                    print(f"  ❌ [DUPLICATE_CHECK] FAIL: {dup_count} duplicate primary key sets on [{merge_key}]")
                
                validation_records.append({
                    "table_id": table_id,
                    "table_name": target_table_name,
                    "batch_id": int(batch_num) if batch_num else None,
                    "check_type": "DUPLICATE_CHECK",
                    "column_checked": merge_key,
                    "source_value": "0",
                    "target_value": str(dup_count),
                    "status": dup_status,
                    "comment": dup_comment
                })
            except Exception as e:
                logger.error(f"Duplicate check error for {target_table}: {e}")
                validation_records.append({
                    "table_id": table_id,
                    "table_name": target_table_name,
                    "batch_id": int(batch_num) if batch_num else None,
                    "check_type": "DUPLICATE_CHECK",
                    "column_checked": merge_key,
                    "source_value": "0",
                    "target_value": "ERROR",
                    "status": "FAIL",
                    "comment": f"Error executing duplicate check: {str(e)[:250]}"
                })

finally:
    if connection:
        connection.close()

# Write validation results to the audit log table
if validation_records:
    pdf_results = pd.DataFrame(validation_records)
    results_df = spark.createDataFrame(pdf_results)
    results_df = results_df.withColumn("execution_time", current_timestamp())
    
    # Append test results to audit table
    results_df.write.format("delta").mode("append").saveAsTable("spark_training.metadata_schema.bronze_validation_log")
    logger.info(f"Successfully recorded {len(validation_records)} test results to 'spark_training.metadata_schema.bronze_validation_log'.")
    print(f"\nSuccessfully recorded {len(validation_records)} validation checks to 'spark_training.metadata_schema.bronze_validation_log'.")

In [ ]:
# 8. Evaluation, Results Display, and Databricks Workflow Failure Alert
print("\n" + "="*80)
print("=== BRONZE DATA QUALITY & RECONCILIATION AUDIT REPORT ===")
print("="*80)

# Display the most recent validation results directly in notebook output
summary_df = spark.sql("""
    SELECT 
        table_id,
        table_name,
        batch_id,
        check_type,
        column_checked,
        source_value,
        target_value,
        status,
        comment
    FROM spark_training.metadata_schema.bronze_validation_log
    WHERE execution_time >= (SELECT MAX(execution_time) - INTERVAL 10 MINUTES FROM spark_training.metadata_schema.bronze_validation_log)
    ORDER BY batch_id, table_id, check_type;
""")
display(summary_df)

# Evaluate if any check failed
failed_checks = [r for r in validation_records if r["status"] == "FAIL"]
total_checks = len(validation_records)
passed_checks = total_checks - len(failed_checks)

print(f"\nSummary: Total Checks: {total_checks} | Passed: {passed_checks} | Failed: {len(failed_checks)}")

if failed_checks:
    failure_messages = [
        f"  • Table ID {r['table_id']} ('{r['table_name']}') [{r['check_type']}]: {r['comment']}"
        for r in failed_checks
    ]
    error_summary = "\n".join(failure_messages)
    logger.error(f"Data Validation FAILED for {len(failed_checks)} checks:\n{error_summary}")
    print(f"\n❌ PIPELINE HALTED: {len(failed_checks)} validation checks failed!")
    print(error_summary)
    
    # Raising this exception marks the Databricks task as FAILED and triggers the Databricks UI email notification
    raise Exception(
        f"DATA VALIDATION FAILED: {len(failed_checks)} of {total_checks} checks failed in Bronze layer!\n"
        f"Audit details logged in 'spark_training.metadata_schema.bronze_validation_log'.\n"
        f"Failures:\n{error_summary}"
    )
else:
    logger.info(f"All {total_checks} data validation checks PASSED successfully!")
    print(f"\n✅ ALL {total_checks} DATA VALIDATION CHECKS PASSED SUCCESSFULLY!")
    print("Bronze layer is verified clean and reconciled with source MySQL database.")